# 05 — Baseline Models (Stage 7)
`DummyClassifier(strategy='most_frequent')` sets the floor; Logistic Regression (standardised inputs, small GridSearchCV over C and class_weight) is a simple ML baseline. Cross-validation results only; test results are in notebook 10.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
FIG = ROOT / 'figures'; REP = ROOT / 'reports'

## Dummy classifier

In [2]:
summary = pd.read_csv(REP / 'tuning' / 'tuning_summary.csv').set_index('model')
row = summary.loc['dummy']
print('Search             :', row['search'], f"({row['n_candidates']} candidates)")
print('CV Macro F1        :', f"{row['cv_macro_f1_mean']:.4f} ± {row['cv_macro_f1_std']:.4f}")
print('Train Macro F1     :', f"{row['train_macro_f1_mean']:.4f}")
print('Train - CV gap     :', f"{row['train_minus_cv_gap']:+.4f}")
print('Best parameters    :', row['best_params'])

Search             : none (fixed strategy) (1 candidates)
CV Macro F1        : 0.1969 ± 0.0000
Train Macro F1     : 0.1969
Train - CV gap     : +0.0000
Best parameters    : {}


Fold-wise Macro F1 (same 5 stratified folds for every model):

In [3]:
folds = pd.read_csv(REP / 'tuning' / 'cv_fold_scores.csv', index_col=0)
folds.loc[[i for i in folds.index if i.startswith('Baseline (Dummy')]]

,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
"Baseline (Dummy, most frequent)",0.1969,0.1969,0.1969,0.1969,0.1969,0.1969,0.0


**Why Macro F1 matters.** The dummy classifier's F1 for No Failure is ~0.98, but 0 for every failure class, so Macro F1 (the unweighted mean over 5 classes) is ~0.197 while accuracy would be ~0.97.

## Logistic Regression

In [4]:
summary = pd.read_csv(REP / 'tuning' / 'tuning_summary.csv').set_index('model')
row = summary.loc['logistic_regression']
print('Search             :', row['search'], f"({row['n_candidates']} candidates)")
print('CV Macro F1        :', f"{row['cv_macro_f1_mean']:.4f} ± {row['cv_macro_f1_std']:.4f}")
print('Train Macro F1     :', f"{row['train_macro_f1_mean']:.4f}")
print('Train - CV gap     :', f"{row['train_minus_cv_gap']:+.4f}")
print('Best parameters    :', row['best_params'])

Search             : GridSearchCV (10 candidates)
CV Macro F1        : 0.7084 ± 0.0419
Train Macro F1     : 0.7143
Train - CV gap     : +0.0060
Best parameters    : {'model__C': 10.0, 'model__class_weight': None}


Fold-wise Macro F1 (same 5 stratified folds for every model):

In [5]:
folds = pd.read_csv(REP / 'tuning' / 'cv_fold_scores.csv', index_col=0)
folds.loc[[i for i in folds.index if i.startswith('Baseline (Logistic')]]

,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
Baseline (Logistic Regression),0.7867,0.6784,0.7176,0.6818,0.6772,0.7084,0.0419


Top 10 candidates of the search, ranked by mean CV Macro F1:

In [6]:
res = pd.read_csv(REP / 'tuning' / 'logistic_regression_search_results.csv')
cols = [c for c in res.columns if c.startswith('param_')] + ['mean_test_score', 'std_test_score', 'mean_train_score', 'mean_fit_time']
res.sort_values('rank_test_score')[cols].head(10).round(4)

,param_model__C,param_model__class_weight,mean_test_score,std_test_score,mean_train_score,mean_fit_time
6,10.00,NaN,0.7084,0.0419,0.7143,0.1009
8,100.00,NaN,0.7069,0.0425,0.7243,0.1118
4,1.00,NaN,0.6390,0.0613,0.6694,0.0720
9,100.00,balanced,0.5722,0.0134,0.5778,0.2123
7,10.00,balanced,0.5405,0.0127,0.5559,0.2104
5,1.00,balanced,0.5059,0.0178,0.5151,0.1367
2,0.10,NaN,0.4574,0.0387,0.4843,0.0461
3,0.10,balanced,0.4402,0.0093,0.4488,0.0853
1,0.01,balanced,0.3606,0.0100,0.3654,0.0548
0,0.01,NaN,0.2030,0.0123,0.2032,0.0319
